# Communal Level EDA & Feature Preparation for Predictive Modeling


This notebook marks the second phase of our exploratory data analysis, shifting focus from the high-level regional overview to the granular communal level. With over 1500 communes, this dataset provides the statistical power required for robust machine learning.


**objectives:**

- investigate the relationships between our features (socio-economic indicators) and the target variable (The Communal Vulnerability Index) to identify the most promising predictors.
- Feature Engineering and Selection : select a final optimized set of features to be used the following modwl phase

## 1-Setup and data loading

**Imports**:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

In [2]:
sns.set_style('whitegrid')#setting plot style

**data loading and filtering**:

In [3]:
df = pd.read_csv("../data/df_final_dataset.csv")

#filtering
territory_type_col= 'Collectivités territoriales'
keyword = 'Commune'
print(f"\nFiltering for rows where the '{territory_type_col}' column contains the keyword: '{keyword}'")

df_communal=df[df[territory_type_col].str.contains(keyword, na=False)].copy()


print(f"Successfully loaded and filtered data. Shape of communal DataFrame: {df_communal.shape}")
df_communal.head()


Filtering for rows where the 'Collectivités territoriales' column contains the keyword: 'Commune'
Successfully loaded and filtered data. Shape of communal DataFrame: (1503, 644)


,Code géographique,Collectivités territoriales,Population légale_ENSEMBLE_TOTAL,Population municipale_ENSEMBLE_TOTAL,Sexe (%)_Masculin_ENSEMBLE_TOTAL,Sexe (%)_Féminin_ENSEMBLE_TOTAL,Âge quinquennal (%)_0-4 ans_ENSEMBLE_TOTAL,Âge quinquennal (%)_5-9 ans_ENSEMBLE_TOTAL,Âge quinquennal (%)_10-14 ans_ENSEMBLE_TOTAL,Âge quinquennal (%)_15-19 ans_ENSEMBLE_TOTAL,...,Mode d’évacuation des déchets ménagers (%)_Bac à ordures de la commune_MENAGES_RURALE,Mode d’évacuation des déchets ménagers (%)_Camion de la commune / Camion privé_MENAGES_RURALE,Mode d’évacuation des déchets ménagers (%)_Dans la nature_MENAGES_RURALE,Mode d’évacuation des déchets ménagers (%)_Autre_MENAGES_RURALE,Combustible de cuisson utilisé (%)_Gaz_MENAGES_RURALE,Combustible de cuisson utilisé (%)_Électricité_MENAGES_RURALE,Combustible de cuisson utilisé (%)_Charbon_MENAGES_RURALE,Combustible de cuisson utilisé (%)_Bois énergie_MENAGES_RURALE,Combustible de cuisson utilisé (%)_Autre_MENAGES_RURALE,Distance moyenne des logements à la route goudronnée (Km)_MENAGES_RURALE
2,01.511.01.0,Commune de Tanger,1275428,1272718,50.6,49.4,8.3,9.2,8.6,7.4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,01.511.01.01,Commune d'Assilah,36039,33975,50.2,49.8,7.3,8.6,8.4,7.2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,01.511.01.09,Commune de Gueznaia,88676,88673,51.4,48.6,10.9,11.5,9.7,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,01.511.05.19,Commune de Hjar Ennhal,27204,25301,52.2,47.8,11.1,12.0,11.2,7.5,...,46.7,1.4,37.7,14.2,99.2,0.4,0.2,0.0,0.1,0.6
11,01.511.05.07,Commune de Dar Chaoui,4471,4469,53.2,46.8,7.3,8.4,10.2,9.4,...,0.0,0.0,100.0,0.0,83.6,0.1,1.0,14.9,0.3,0.5


## 2-Constructing the target variable - the communal vulnerability index

In [4]:
vulnerability_indicators = ["Taux d'analphabétisme des 15 ans et plus (%)_ENSEMBLE_TOTAL",
                            "Niveau d'études dans l'enseignement général (%)_Supérieur_ENSEMBLE_TOTAL",
                            "Niveau d'études dans l'enseignement général (%)_Aucun niveau d'études_ENSEMBLE_TOTAL",
                            "Taux de chômage (%)_ENSEMBLE_TOTAL",
                            "Statut professionnel des actifs occupés de 15 ans et plus (%)_Indépendant_ENSEMBLE_TOTAL",
                            "Statut professionnel des actifs occupés de 15 ans et plus (%)_Aide familial_ENSEMBLE_TOTAL",
                            "Type de logement (%)_Maison sommaire / Bidonville_MENAGES_TOTAL",
                            "Disponibilité des éléments essentiels de confort (%)_Cuisine_MENAGES_TOTAL",
                            "Disponibilité des éléments essentiels de confort (%)_ W.-C._MENAGES_TOTAL",
                            "Disponibilité des éléments essentiels de confort (%)_Pièce d'eau_MENAGES_TOTAL",
                            "Disponibilité des éléments essentiels de confort (%)_Électricité_MENAGES_TOTAL",
                            "Disponibilité des éléments essentiels de confort (%)_Eau courante_MENAGES_TOTAL",
                            "Mode d’évacuation des déchets ménagers (%)_Dans la nature_MENAGES_TOTAL",
                            "Distance moyenne des logements à la route goudronnée (Km)_MENAGES_TOTAL",
                            "Combustible de cuisson utilisé (%)_Charbon_MENAGES_TOTAL"]

df_for_pca = df_communal[vulnerability_indicators].copy()

    

- **pca score function**

In [5]:
def get_pca_score(df):
    """
    Takes a dataframe, scales it, performs PCA, and returns the first principal component score.
    """
    
    imputer = SimpleImputer(strategy='mean')
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(imputer.fit_transform(df))
    
    
    pca = PCA(n_components=1)
    principal_components = pca.fit_transform(data_scaled)#transform=shadow position
    
    
    return principal_components.flatten(), pca

In [6]:
vulnerability_score, pca_vulnerability = get_pca_score(df_for_pca)

- **create the target variable 'y'**

In [7]:
y = pd.Series(vulnerability_score, index=df_communal.index, name="Communal_Vulnerability_Index")

In [8]:
#checking the direction of the new vulnerability index( a higher score should mean higher vulnerability
illiteracy_col = "Taux d'analphabétisme des 15 ans et plus (%)_ENSEMBLE_TOTAL"
loadings_vulnerability = pd.Series(pca_vulnerability.components_[0], index=df_for_pca.columns)

illiteracy_loading = loadings_vulnerability.get(illiteracy_col, 0)

print(f"The loading for '{illiteracy_col}' is: {illiteracy_loading:.3f}")
if illiteracy_loading < 0:
    print("Flipping sign: A higher score will now correctly mean HIGHER Vulnerability.")
    y = -y  # Flip the entire 'y' Series by multiplying by -1
else:
    print("Sign is correct: A higher score already means HIGHER Vulnerability.")

print("\n--- Communal Vulnerability Index (Our Target 'y') is now finalized and correctly oriented ---")



The loading for 'Taux d'analphabétisme des 15 ans et plus (%)_ENSEMBLE_TOTAL' is: 0.380
Sign is correct: A higher score already means HIGHER Vulnerability.

--- Communal Vulnerability Index (Our Target 'y') is now finalized and correctly oriented ---


count    1.503000e+03
mean     2.836498e-17
std      2.250016e+00
min     -8.864002e+00
25%     -1.393538e+00
50%      2.185209e-01
75%      1.485486e+00
max      8.844166e+00
Name: Communal_Vulnerability_Index, dtype: float64

## 3-Create the final feature matrix X

In [12]:
used_indicators = vulnerability_indicators 

all_numeric_cols = df_communal.select_dtypes(include=np.number).columns

#new feature set X is all numeric columns minus the ones used (vulnerability indicators)
potential_features = [col for col in all_numeric_cols if col not in used_indicators and col != y.name]

X= df_communal[potential_features].copy()
print(f"Created feature matrix X with {X.shape[1]} potential predictors.")


Created feature matrix X with 627 potential predictors.


## 4- Descriptive Statistics

- **Target variable (y)**

In [13]:
print("Descriptive Statistics for the Communal Vulnerability Index (y)")
display(y.describe())

Descriptive Statistics for the Communal Vulnerability Index (y)


count    1.503000e+03
mean     2.836498e-17
std      2.250016e+00
min     -8.864002e+00
25%     -1.393538e+00
50%      2.185209e-01
75%      1.485486e+00
max      8.844166e+00
Name: Communal_Vulnerability_Index, dtype: float64

- the mean is close to 0 as expected from PCA on scaled data
- std indicates a healthy amount of variation in the scores. It's not too narrow and not too wide
- min and max wide has a clear wide range from approximately -8.9 to +8.8
-  the median (0.218) is slightly higher than the mean (0). This suggests that the distribution is slightly skewed.


In [15]:
# --- STEP 2: Select the Best Features (X) ---

# Define X as all other numeric columns...
X = df_communal.select_dtypes(include=np.number).drop(columns=vulnerability_indicators)

# Calculate correlations with y...
correlations_with_target = X.corrwith(y).abs().sort_values(ascending=False)

# Select the top 40 predictors...
top_predictors = correlations_with_target.head(40).index
X_top = X[top_predictors]

# Prune for multicollinearity...
# (Use your heatmap and prune_correlated_features function on X_top)
final_feature_names = prune_correlated_features(X_top.corr(), threshold=0.9)
X_final = X[final_feature_names]

print(f"--- Final Feature Matrix 'X_final' with {X_final.shape[1]} features is complete. ---")


C:\Users\fadwa\anaconda3\Lib\site-packages\numpy\lib\_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\fadwa\anaconda3\Lib\site-packages\numpy\lib\_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


NameError: name 'prune_correlated_features' is not defined